In [1]:
!python3 -m pip install pyarango
!python3 -m pip install "python-arango>=5.0" 


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import requests
import sys
import time                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

from arango.client import ArangoClient


# Initialize the client for ArangoDB.
client = ArangoClient(hosts="http://localhost:8529")


# Connect to "db_test" database as root user.
db = client.db("db_test", username="root", password="test")       

aql = db.aql

## Locations Data
Let us create a collection with some filming locations for Games of Thrones.

![locations](https://github.com/arangodb/interactive_tutorials/blob/master/notebooks/img/Locations_Map.png?raw=1)

In [3]:
if not db.has_collection(name="Locations"):
    db.create_collection(name="Locations")

In [5]:
insert_query = """
LET places = [
    { "name": "Dragonstone", "coordinate": [ 55.167801, -6.815096 ] },
    { "name": "King's Landing", "coordinate": [ 42.639752, 18.110189 ] },
    { "name": "The Red Keep", "coordinate": [ 35.896447, 14.446442 ] },
    { "name": "Yunkai", "coordinate": [ 31.046642, -7.129532 ] },
    { "name": "Astapor", "coordinate": [ 31.50974, -9.774249 ] },
    { "name": "Winterfell", "coordinate": [ 54.368321, -5.581312 ] },
    { "name": "Vaes Dothrak", "coordinate": [ 54.16776, -6.096125 ] },
    { "name": "Beyond the wall", "coordinate": [ 64.265473, -21.094093 ] }
]


FOR place IN places
    INSERT place INTO Locations
"""

aql.execute(insert_query)

<Cursor>

As before let us check the `Locations` collection:

In [ ]:
all_locations_names = """
FOR p IN Locations
    RETURN p.name
"""

query_result = aql.execute(all_locations_names)
for doc in  query_result:
    print(doc)
    print()

Dragonstone

King's Landing

The Red Keep

Yunkai

Astapor

Winterfell

Vaes Dothrak

Beyond the wall



## Geospatial Index

To query based on coordinates, a [geo index](https://www.arangodb.com/docs/stable/indexing-geo.html) is required. It determines which fields contain the latitude and longitude values.

In [4]:
index_result = db.collection(name="Locations").add_index({ "type": "geo", "fields": ["coordinate"]})
print(index_result)

{'bestIndexedLevel': 17, 'fields': ['coordinate'], 'geoJson': False, 'id': '35463', 'isNewlyCreated': True, 'legacyPolygons': False, 'maxNumCoverCells': 8, 'name': 'idx_1864518826826137600', 'sparse': True, 'type': 'geo', 'unique': False, 'worstIndexedLevel': 4}


## Find nearby Locations
A `FOR` loop is used to iterate over the results of a function call to NEAR() to find the n closest coordinates to a reference point, and return the documents with the nearby locations. The default for n is 100, which means 100 documents are returned at most, the closest matches first.

In below example, the limit is set to 3. The origin (the reference point) is a coordinate somewhere downtown in Dublin, Ireland:

In [5]:
near_locations_names = """
FOR loc IN NEAR(Locations, 53.35, -6.26, 3)
    RETURN {
        name: loc.name,
        latitude: loc.coordinate[0],
        longitude: loc.coordinate[1]
    }
"""

query_result = aql.execute(near_locations_names)
for doc in  query_result:
    print(doc)
    print() 

{'name': 'Vaes Dothrak', 'latitude': 54.16776, 'longitude': -6.096125}

{'name': 'Winterfell', 'latitude': 54.368321, 'longitude': -5.581312}

{'name': 'Dragonstone', 'latitude': 55.167801, 'longitude': -6.815096}



## Find locations within radius
Instead of `NEAR()` we can also use `WITHIN()`, to search for locations within a given radius from a reference point. The syntax is the same as for `NEAR()`, except for the fourth parameter, which specifies the radius instead of a limit. The unit for the radius is meters. The example uses a radius of 200,000 meters (200 kilometers):

In [6]:
within_locations_names = """
FOR loc IN WITHIN(Locations, 53.35, -6.26, 200 * 1000)
    RETURN {
        name: loc.name,
        latitude: loc.coordinate[0],
        longitude: loc.coordinate[1]
    }
"""

query_result = aql.execute(within_locations_names)
for doc in  query_result:
    print(doc)
    print() 

{'name': 'Vaes Dothrak', 'latitude': 54.16776, 'longitude': -6.096125}

{'name': 'Winterfell', 'latitude': 54.368321, 'longitude': -5.581312}



## Calculating the Distance

Both `NEAR()` and `WITHIN()` can return the distance to the reference point by adding an optional fifth parameter. It has to be a string, which will be used as attribute name for an additional attribute with the distance in meters:

In [8]:
near_locations_names = """
FOR loc IN NEAR(Locations, 53.35, -6.26, 3, "distance")
    RETURN {
        name: loc.name,
        latitude: loc.coordinate[0],
        longitude: loc.coordinate[1],
        distance: loc.distance / 1000
    }
"""

query_result = aql.execute(near_locations_names)
for doc in  query_result:
    print(doc)
    print()  

{'name': 'Vaes Dothrak', 'latitude': 54.16776, 'longitude': -6.096125, 'distance': 91.56658640314484}

{'name': 'Winterfell', 'latitude': 54.368321, 'longitude': -5.581312, 'distance': 121.66399816395003}

{'name': 'Dragonstone', 'latitude': 55.167801, 'longitude': -6.815096, 'distance': 205.31879386198273}

